In [ ]:
# Libraries ----
import os
import re
import sys
import warnings
import numpy as np  # type: ignore
import pandas as pd  # type: ignore

sys.path.append("../modules")
import misc_functions as mf  # type: ignore
import plot_interactive as pi  # type: ignore
import estimate_hoi_measures as ehm  # type: ignore
import estimate_complexity_measures as ecm  # type: ignore
import estimate_synchronization_analysis as esa  # type: ignore
import estimate_complex_network_analysis as ecna  # type: ignore

# Global options ----
warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None
pd.set_option("display.max_columns", None)

## Global variables

In [2]:
log_path = "../logs"
input_path = "../input_files"
output_path = "../output_files"
input_generation_date = "2025-02-18"
x_bounds = [0, 1920]
y_bounds = [0, 1080]
t_threshold = 3600

## Load and prepare data of all tracked videos

In [3]:
cols = [
    "particles",
    "video",
    "permuted_id",
    "time",
    "position_x",
    "position_y",
    "corrected_orientation",
    "n_x",
    "n_y",
    "n_orientation",
    "norm"
]

df_final = []
for file in os.listdir(input_path):
    df = pd.read_csv(input_path + "/" + file, low_memory=False)
    df["video"] = re.sub("reviewed_", "", re.sub(".csv", "", file))
    df["particles"] = df["video"].str[0]
    df["n_x"] = df["position_x"] / x_bounds[1] - 0.5  # Normalized position in X-axis
    df["n_y"] = df["position_y"] / y_bounds[1] - 0.5  # Normalized position in Y-axis
    df["n_orientation"] = np.sin(df["corrected_orientation"])  # Normalized Orientation
    df["norm"] = (
        np.power(df["n_x"], 2)
        + np.power(df["n_y"], 2)
        + np.power(df["n_orientation"], 2)
    )
    df_final.append(df[df["time"] <= t_threshold][cols])

df_final = pd.concat(df_final, ignore_index=True)
df_final

,particles,video,permuted_id,time,position_x,position_y,corrected_orientation,n_x,n_y,n_orientation,norm
0,2,2n_0m_2f_230330_1,0,0,1657.350058,59.069803,1.473471,0.363203,-0.445306,0.995268,1.320771
1,2,2n_0m_2f_230330_1,1,0,1221.582078,92.427967,-1.507389,0.136241,-0.414419,-0.997990,1.186289
2,2,2n_0m_2f_230330_1,0,3,1675.103082,58.309940,1.546517,0.372450,-0.446009,0.999705,1.337054
3,2,2n_0m_2f_230330_1,1,3,1221.530021,92.132771,-1.512559,0.136214,-0.414692,-0.998305,1.187136
4,2,2n_0m_2f_230330_1,0,6,1675.049035,58.351203,1.544728,0.372421,-0.445971,0.999660,1.336909
...,...,...,...,...,...,...,...,...,...,...,...
54040,4,4n_4m_0f_230523_2,3,3597,411.256103,541.839636,-0.028980,-0.285804,0.001703,-0.028976,0.082527
54041,4,4n_4m_0f_230523_2,0,3600,244.905086,574.666178,3.103433,-0.372445,0.032098,0.038151,0.141201
54042,4,4n_4m_0f_230523_2,1,3600,315.592605,783.672704,2.764931,-0.335629,0.225623,0.367818,0.298842
54043,4,4n_4m_0f_230523_2,2,3600,923.548563,321.790707,0.997767,-0.018985,-0.202046,0.840262,0.747223


## Estimate elementary symmetric polynomials

In [4]:
cols = [
    "particles",
    "video",
    "permuted_id",
    "time",
    "position_x",
    "position_y",
    "corrected_orientation",
    "n_x",
    "n_y",
    "n_orientation",
    "norm"
]

df_esp = []
for video in df_final["video"].unique():
    df_aux = mf.estimate_esp(
        df=df_final[df_final["video"] == video],
        filter_step=None
    ).rename(columns={"order": "permuted_id"})
    df_aux["video"] = video
    df_aux["particles"] = df_aux["video"].str[0]
    df_aux["position_x"] = df_aux["n_x"]
    df_aux["position_y"] = df_aux["n_y"]
    df_aux["n_orientation"] = np.sin(df_aux["corrected_orientation"])  # Normalized Orientation
    df_aux["norm"] = (
        np.power(df_aux["n_x"], 2)
        + np.power(df_aux["n_y"], 2)
        + np.power(df_aux["n_orientation"], 2)
    )
    
    df_esp.append(df_aux[cols])

df_esp = pd.concat(df_esp, ignore_index=True)
df_esp

Estimate elementary symmetric polynomials for: 2 ids
Estimate elementary symmetric polynomials for: 2 ids
Estimate elementary symmetric polynomials for: 2 ids
Estimate elementary symmetric polynomials for: 2 ids
Estimate elementary symmetric polynomials for: 2 ids
Estimate elementary symmetric polynomials for: 2 ids
Estimate elementary symmetric polynomials for: 2 ids
Estimate elementary symmetric polynomials for: 2 ids
Estimate elementary symmetric polynomials for: 2 ids
Estimate elementary symmetric polynomials for: 2 ids
Estimate elementary symmetric polynomials for: 3 ids
Estimate elementary symmetric polynomials for: 3 ids
Estimate elementary symmetric polynomials for: 3 ids
Estimate elementary symmetric polynomials for: 3 ids
Estimate elementary symmetric polynomials for: 3 ids
Estimate elementary symmetric polynomials for: 3 ids
Estimate elementary symmetric polynomials for: 3 ids
Estimate elementary symmetric polynomials for: 4 ids


,particles,video,permuted_id,time,position_x,position_y,corrected_orientation,n_x,n_y,n_orientation,norm
0,2,2n_0m_2f_230330_1,0,0,1.499444,0.140276,1.473471,1.499444,0.140276,0.995268,3.258567
1,2,2n_0m_2f_230330_1,0,3,1.508663,0.139299,1.546517,1.508663,0.139299,0.999705,3.294879
2,2,2n_0m_2f_230330_1,0,6,1.508599,0.139346,1.544728,1.508599,0.139346,0.999660,3.294610
3,2,2n_0m_2f_230330_1,0,9,1.499593,0.140113,1.484579,1.499593,0.140113,0.996286,3.260995
4,2,2n_0m_2f_230330_1,0,12,1.499590,0.140080,1.483964,1.499590,0.140080,0.996232,3.260873
...,...,...,...,...,...,...,...,...,...,...,...
54040,4,4n_4m_0f_230523_2,3,3588,0.001911,0.075984,2.859093,0.001911,0.075984,0.278757,0.083483
54041,4,4n_4m_0f_230523_2,3,3591,0.002196,0.063984,-0.176111,0.002196,0.063984,-0.175202,0.034795
54042,4,4n_4m_0f_230523_2,3,3594,0.002227,0.063950,-0.121110,0.002227,0.063950,-0.120815,0.018691
54043,4,4n_4m_0f_230523_2,3,3597,0.002101,0.063481,-0.028980,0.002101,0.063481,-0.028976,0.004874


## Plot time series of each individual

In [5]:
pi.interactive_plot(df=df_final, interval_size=600)
pi.interactive_plot(df=df_esp, interval_size=600)

# Are there higher-order interactions in 3 and 4 cockroach videos?

- Aim: To classify regimes as redundancy-dominated or synergy-dominated over time.
- Data: Distances from the center of the box and orientation of the cockroaches respect to the vertical.

## Get O-information data between individuals
O-Information is a measure used in information theory to quantify **high-order interactions** in multivariate systems. It distinguishes between redundant and synergistic information sharing across multiple variables.

### **Mathematical Formulation**

For a set of random variables $(X_1, X_2, ..., X_N)$, the O-Information and exogenous information are defined as:

\begin{align}
    \Omega(X_1, X_2, ..., X_N) &= TC(\mathbf{X}_{n}) - DTC(\mathbf{X}_{n}) = \sum_{i=1}^{N} I(X_i; \mathbf{X}_{-i}) - I(\mathbf{X}_1, ..., \mathbf{X}_N) \\
    S(X_1, X_2, ..., X_N) &= TC(\mathbf{X}_{n}) + DTC(\mathbf{X}_{n})
\end{align}

respectively, where:

- $I(X_i; \mathbf{X}_{-i})$ is the mutual information between $X_i$ and the rest.
- $I(\mathbf{X}_1, ..., \mathbf{X}_N)$ is the total multivariate mutual information.
- If $\Omega>0$, the system is redundancy-dominated.
- If $\Omega<0$, the system is synergy-dominated.


In [6]:
window_sizes = [
    [30], [36], [45], [48], [60], [72], [75], [90],
    [120], [144], [150], [180], [225],
    [240], [300], [360], [450], [600],
    [720], [900], [1200], [1800], [t_threshold]
]
df_oinfo = []
for video in df_esp["video"].unique():
    df = df_esp[df_esp["video"] == video]
    if int(video[0]) >= 3:
        df_aux = ehm.estimate_oinfo_multiple_windows(
            df=df,
            window_sizes=window_sizes,
            log_path=log_path,
            log_filename="log_hoi_esp",
            verbose=1,
            tqdm_bar=True
        )
        df_oinfo.append(df_aux)

df_oinfo = pd.concat(df_oinfo, ignore_index=True)
df_oinfo.to_csv(output_path + "/df_hoi_esp.csv", index=False)
df_oinfo

100%|███████████████████████| 23/23 [06:03<00:00, 15.81s/it]


,video,t_range,size,multiplet,oinfo_distance,oinfo_orientation,sinfo_distance,sinfo_orientation
0,3n_0m_3f_230404_1,0 - 30,30,012,0.032150,0.028209,4.896884,0.170158
1,3n_0m_3f_230404_1,30 - 60,30,012,NaN,0.187642,NaN,2.382899
2,3n_0m_3f_230404_1,60 - 90,30,012,NaN,0.100093,NaN,0.780154
3,3n_0m_3f_230404_1,90 - 120,30,012,1.532181,-0.054813,7.081939,0.305895
4,3n_0m_3f_230404_1,120 - 150,30,012,1.697380,0.066044,7.658750,-0.021222
...,...,...,...,...,...,...,...,...
9043,4n_4m_0f_230523_2,0 - 3600,3600,012,1.066933,-0.012133,5.673085,0.355206
9044,4n_4m_0f_230523_2,0 - 3600,3600,013,0.705347,-0.001674,6.091899,0.426130
9045,4n_4m_0f_230523_2,0 - 3600,3600,023,0.878658,-0.012717,4.565824,0.289693
9046,4n_4m_0f_230523_2,0 - 3600,3600,123,1.575185,0.009411,6.046204,0.498178


---
# How different is the behavior when sex ratio changes (fixed group size)?

- Aim: To estimate Hurst exponent (asses persistence), Permutation entropy (predictability), Statistical complexity (assess the balance of order and randomness) for each cockroach’s time series (distance, orientation) → persistence vs. randomness.
- Data: Distances from the center of the box and orientation of the cockroaches respect to the vertical.
---

## Get Hurst exponent, permutation entropy and Statistical complexity data between individuals

### **Mathematical Formulation**

#### 1. Hurst Exponent

The Hurst exponent ($H$) is a statistical measure used to evaluate the long-term memory of time series data. It quantifies the tendency of a time series to either:

- Persist in its trend ($H > 0.5$)
- Exhibit a random walk ($H ≈ 0.5$)
- Mean-revert (antipersistence) ($H < 0.5$)

The Hurst exponent is estimated using the rescaled range (R/S) analysis:

\begin{equation}
    H = \lim_{T \to \infty} \frac{\log(R/S)}{\log(T)},
\end{equation}

where $R$ is the range of cumulative deviations from the mean, $S$ is the standard deviation, and $T$ is the time window size.

Also, Multifractal Detrended Fluctuation Analysis (MF-DFA) is an extension of DFA (Detrended Fluctuation Analysis) that measures the **multifractal properties** of a time series by analyzing its scaling behavior at different moments $q$. The fluctuation function is defined as:

\begin{equation}
    F_q(s) = \left( \frac{1}{N_s} \sum_{\nu=1}^{N_s} F^q(\nu, s) \right)^{\frac{1}{q}}
\end{equation}

where $F(\nu, s)$ is the local detrended fluctuation at segment $\nu$, and $s$ is the window size.

#### 2. Permutation Entropy

Permutation Entropy ($PE$) is a nonlinear measure of time series complexity introduced by Bandt & Pompe (2002). It quantifies the randomness in a time series by analyzing the frequency of ordinal patterns of a given length.

Given a time series $X = \{x_1, x_2, ..., x_N\}$ and an embedding dimension $d$, we extract ordinal patterns by ranking the values within sliding windows of size $\tau$. The permutation entropy is then computed as:

\begin{equation}
    H_p = - \sum p(\pi) \log p(\pi)
\end{equation}

where $p(\pi)$ is the probability of each ordinal pattern $\pi$.

#### 3. Statistical complexity

Statistical complexity measures the balance between disorder and structure in a system. It complements entropy by identifying structured patterns within randomness.

A widely used definition is the **Jensen-Shannon complexity**, which combines permutation entropy and disequilibrium:

\begin{equation}
    C_J = H_p \cdot Q_J
\end{equation}

where $Q_J$ is the Jensen-Shannon divergence measuring disequilibrium.

---
## **References**

- Bandt, C., & Pompe, B. (2002). Permutation entropy: A natural complexity measure for time series.
- Rosso, O. A., et al. (2007). Distinguishing noise from chaos.
- Ribeiro, H. V., et al. (2012). Characterizing time series through complexity-entropy curves.
- `ordpy`: A Python library for ordinal pattern analysis.
---

In [11]:
window_sizes = [
    # [30], [36], [45], [48], [60], [72], [75], [90],
    [120], [144], [150], [180], [225], [240],
    [300], [360], [450], [600],
    [720], [900], [1200], [1800], [t_threshold]
]
df_complexity = []
for video in df_esp["video"].unique():
    df = df_esp[df_esp["video"] == video]
    df_aux = ecm.estimate_multiple_hurst_complexity(
        df=df,
        window_sizes=window_sizes,
        q=2,
        dx=3,
        taux=1,
        log_path=log_path,
        log_filename="log_complexity_esp",
        verbose=1,
        tqdm_bar=True
    )

    df_complexity.append(df_aux)

df_complexity = pd.concat(df_complexity, ignore_index=True)
df_complexity.to_csv(output_path + "/df_complexity_esp.csv", index=False)
df_complexity

100%|███████████████████████| 15/15 [00:06<00:00,  2.34it/s]


,video,t_range,size,permuted_id,H_distance,PE_distance,C_distance,H_orientation,PE_orientation,C_orientation
0,2n_0m_2f_230330_1,0 - 120,120,0,1.029797,0.979728,0.021444,0.505706,0.943277,0.052641
1,2n_0m_2f_230330_1,0 - 120,120,1,0.959033,0.965997,0.033541,0.980146,0.964793,0.033099
2,2n_0m_2f_230330_1,120 - 240,120,0,0.942687,0.974891,0.023966,0.342645,0.968650,0.029637
3,2n_0m_2f_230330_1,120 - 240,120,1,0.816026,0.969987,0.030004,0.523228,0.965623,0.031898
4,2n_0m_2f_230330_1,240 - 360,120,0,0.999692,0.939432,0.055756,0.365565,0.964778,0.031184
...,...,...,...,...,...,...,...,...,...,...
8140,4n_4m_0f_230523_2,1800 - 3600,1800,3,0.649301,0.930434,0.062389,0.343959,0.945015,0.049889
8141,4n_4m_0f_230523_2,0 - 3600,3600,0,0.531686,0.942137,0.052348,0.232728,0.956867,0.039692
8142,4n_4m_0f_230523_2,0 - 3600,3600,1,0.541991,0.944273,0.050585,0.281743,0.913399,0.076053
8143,4n_4m_0f_230523_2,0 - 3600,3600,2,0.571390,0.938372,0.055653,0.400831,0.908576,0.079667


---
# How different is the behavior when sex ratio changes (fixed group size)?

- Aim: To examine coordination and coupling using Cross-recurrence quantification analysis (CRQA).
- Data: Distances from the center of the box and orientation of the cockroaches respect to the vertical.
---

## Synchronization in Complex Systems

### **Mathematical Formulation**

#### 1. Recurrence Plot Analysis (RPA)

**Recurrence Plot Analysis (RPA)** is a nonlinear method for studying the dynamics and synchronization of complex systems. It visualizes when a system revisits the same or similar state in its phase space.

Given a trajectory $\vec{x}_i \in \mathbb{R}^d$, the **recurrence matrix** is defined as:

\begin{equation}
    R_{i,j} = \Theta(\varepsilon - \|\vec{x}_i - \vec{x}_j\|),
\end{equation}

where:
- $\varepsilon$ is a threshold,
- $\Theta$ is the Heaviside step function,
- $R_{i,j} = 1$ means state $i$ recurs at time $j$.

The resulting binary matrix can be visualized as a **recurrence plot**.

**Recurrence Quantification Analysis (RQA)** converts the visual features of recurrence plots into quantitative metrics:

| Metric | Description |
|--------|-------------|
| **RR** – Recurrence Rate | Ratio of recurrent points in the plot |
| **DET** – Determinism | Fraction of recurrent points forming diagonal lines (indicating predictability) |
| **L** – Average Diagonal Line Length | Mean time over which the system exhibits similar behavior |
| **ENTR** – Entropy | Shannon entropy of diagonal line lengths (complexity measure) |
| **LAM** – Laminarity | Fraction of points forming vertical lines (indicating intermittency or stationarity) |
| **TT** – Trapping Time | Average vertical line length |

These metrics help detect patterns such as synchronization, chaos, and transitions in dynamics.

---

#### 2. Kuramoto Model & Synchronization

The **Kuramoto model** is a classical model for understanding synchronization in populations of coupled oscillators (e.g., neurons, fireflies, mechanical rotors).

Each oscillator $i$ has a phase $\theta_i(t)$, and their interaction is governed by:

\begin{equation}
    \frac{d\theta_i}{dt} = \omega_i + \frac{K}{N} \sum_{j=1}^{N} \sin(\theta_j - \theta_i),
\end{equation}

where:
- $\omega_i$ is the natural frequency,
- $K$ is the coupling strength,
- $N$ is the number of oscillators.

The **degree of synchronization** is measured by the **Kuramoto order parameter** $R(t)$:

\begin{equation}
    R(t) = \left| \frac{1}{N} \sum_{j=1}^{N} e^{i \theta_j(t)} \right|
\end{equation}

- $R(t) \in [0, 1]$
- $R(t) \approx 1$: perfect phase synchronization
- $R(t) \approx 0$: desynchronized or incoherent state

This metric summarizes global coherence in the system and is often visualized as a function of time or coupling strength $K$.

The **Kuramoto parameter** offers a global phase measure, while **RQA** captures finer nonlinear structures and local synchrony.

---

In [16]:
Rs, df_metrics = esa.estimate_multiple_crqa(
    df=df_esp,
    epsilon_factor=0.05,
    plot=False,
    min_diagonal_length=2,
    min_vertical_length=2
)
df_metrics.to_csv(output_path + "/df_recurrence_esp.csv", index=False)
df_metrics

,video,id_pair,RR,DET,L,ENTR,LAM,TT,num_diag_lines,num_vert_lines
0,2n_0m_2f_230330_1,00,0.093628,0.581085,2.804181,1.202925,0.765381,3.686831,27985,28036
1,2n_0m_2f_230330_1,01,0.097573,0.611579,2.889034,1.272874,0.784274,3.876313,29793,28475
2,2n_0m_2f_230330_1,11,0.115089,0.658360,3.079748,1.414132,0.810927,4.265598,35487,31559
3,2n_0m_2f_230404_1,00,0.070046,0.945425,12.626702,2.933216,0.973494,20.288160,7565,4848
4,2n_0m_2f_230404_1,01,0.063130,0.958225,14.311137,3.012803,0.979848,22.866222,6097,3902
...,...,...,...,...,...,...,...,...,...,...
77,4n_4m_0f_230523_2,12,0.020550,0.802976,4.157380,1.826186,0.885429,5.284938,5725,4966
78,4n_4m_0f_230523_2,13,0.027021,0.787146,3.937749,1.762875,0.877793,5.133083,7791,6665
79,4n_4m_0f_230523_2,22,0.040998,0.806240,3.853933,1.777435,0.896542,5.169868,12371,10255
80,4n_4m_0f_230523_2,23,0.024544,0.831483,4.181986,1.879418,0.907522,5.476223,7039,5867


In [27]:
# Estimate Kuramoto order parameter
df_synchronization = []
for video in df_esp["video"].unique():
    mask = df_esp["video"] == video
    times, Rs = esa.estimate_kuramoto_order_parameter(df=df_esp[mask])
    df_synchronization.append(pd.DataFrame({
        "video": [video]*len(times),
        "time": times,
        "order_parameter": Rs
    }))
df_synchronization = pd.concat(df_synchronization, ignore_index=True)
df_synchronization.to_csv(output_path + "/df_kuramoto_esp.csv", index=False)

# How different is the behavior with different number of cockroaches?

- Aim: To quantify network properties (modularity, clustering, degree distribution).
- Data: Distances from the center of the box and orientation of the cockroaches respect to the vertical.

## Get Visibility Graph between individuals
The **Visibility Graph** ($VG$) is a method that transforms time series into a complex network, where data points in the series are mapped to nodes and edges are created based on visibility criteria.

### **Mathematical Formulation**
Given a time series $X = \{ x_1, x_2, ..., x_N \}$, two data points $(x_i, t_i)$ and $(x_j, t_j)$ are connected if any intermediate point $(x_k, t_k)$ satisfies:

\begin{equation}
    x_k < x_i + (x_j - x_i) \frac{t_k - t_i}{t_j - t_i}, \quad \forall k \in (i,j)
\end{equation}

In [20]:
window_sizes = [
    [30], [36], [45], [48], [60], [72], [75], [90],
    [120], [144], [150], [180], [225], [240],
    [300], [360], [450], [600],
    [720], [900], [1200], [1800], [t_threshold]
]
df_network_all, df_network_nodes = [], []
for video in df_esp["video"].unique():
    df = df_esp[df_esp["video"] == video]
    df_all, df_nodes = ecna.estimate_multiple_vg(
        df=df,
        window_sizes=window_sizes,
        log_path=log_path,
        log_filename="log_network_esp",
        verbose=1,
        tqdm_bar=True
    )
    df_network_all.append(df_all)
    df_network_nodes.append(df_nodes)

df_network_all = pd.concat(df_network_all, ignore_index=True)
df_network_all.to_csv(output_path + "/df_network_esp.csv", index=False)

df_network_nodes = pd.concat(df_network_nodes, ignore_index=True)
df_network_nodes.to_csv(output_path + "/df_network_nodes_esp.csv", index=False)

100%|███████████████████████| 23/23 [02:03<00:00,  5.39s/it]
